# MCTS à l'inférence — Kamae3

Ce notebook implémente un **MCTS léger** (AlphaZero-style) autour du réseau Kamae3, sans ré-entraînement.

## Principe
- La **policy head** fournit les probabilités a priori `P(s, a)` pour guider l'arbre
- La **value head** évalue les feuilles (pas de rollouts → rapide)
- La formule **PUCT** équilibre exploitation et exploration

## Optimisation pour machine modeste
- Utilise `cancel_last_move()` du Board pour annuler les coups (**pas de deepcopy**)
- Un seul objet Board modifié/restauré → pas d'allocation mémoire en boucle
- `n_simulations` réglable : 50 simulations ≈ jouable en temps réel sur CPU

In [ ]:
import sys
sys.path.append('../onitama/')

import numpy as np
import math
import time
from tqdm import tqdm

from board import Board
from players import Player
from game import Game, GameSession
from dl_players_v6 import CNNPlayer_v6
from players import LookAheadHeuristicPlayer

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
WEIGHTS_PATH   = '../saved-models/Kamae3.weights.h5'

N_SIMULATIONS  = 100     # Nombre de simulations MCTS par coup (50-200 recommandé)
C_PUCT         = 1.5     # Exploration vs exploitation (1.0-2.0)
DIRICHLET_ALPHA = 0.3    # Bruit Dirichlet à la racine (0 = désactivé)
DIRICHLET_EPS   = 0.25   # Poids du bruit Dirichlet (0 = désactivé)

# Benchmark et évaluation
BENCH_N_MOVES  = 50      # Nombre de coups pour le benchmark de vitesse
EVAL_N_GAMES   = 100     # Parties pour l'évaluation finale
# ──────────────────────────────────────────────────────────────────────────────

## Implémentation MCTS

In [ ]:
class MCTSNode:
    """
    Noeud de l'arbre MCTS.

    Convention : W (value_sum) est stocké du point de vue du PARENT
    qui a choisi l'action menant à ce noeud.
    Ainsi Q = W / N représente directement l'utilité de cet enfant
    pour le parent qui sélectionne.
    """
    __slots__ = ['N', 'W', 'prior', 'children']

    def __init__(self, prior: float = 0.0):
        self.N       = 0       # Nombre de visites
        self.W       = 0.0     # Somme des valeurs (du point de vue du parent)
        self.prior   = prior   # P(s, a) depuis le réseau
        self.children = {}     # Action -> MCTSNode

    def q_value(self) -> float:
        return self.W / self.N if self.N > 0 else 0.0

    def puct_score(self, parent_N: int, c_puct: float) -> float:
        """Score PUCT = Q + U, utilisé par le parent pour sélectionner cet enfant."""
        U = c_puct * self.prior * math.sqrt(parent_N) / (1 + self.N)
        return self.q_value() + U

    def is_expanded(self) -> bool:
        return len(self.children) > 0

In [ ]:
class MCTS:
    """
    MCTS léger basé sur le réseau de neurones (AlphaZero-style).

    Optimisation clé : utilise play_move() / cancel_last_move() du Board
    pour traverser l'arbre sans jamais copier l'état — bien plus rapide
    sur CPU que deepcopy.
    """

    def __init__(self, nn_player: CNNPlayer_v6, n_simulations: int = 100,
                 c_puct: float = 1.5,
                 dirichlet_alpha: float = 0.0, dirichlet_eps: float = 0.0):
        self.nn             = nn_player
        self.n_simulations  = n_simulations
        self.c_puct         = c_puct
        self.dir_alpha      = dirichlet_alpha
        self.dir_eps        = dirichlet_eps

    # ── Inférence réseau ───────────────────────────────────────────────────────

    def _nn_evaluate(self, board: Board):
        """Retourne (policy_probs dict{action: prob}, value float) pour l'état courant."""
        import numpy as np, tensorflow as tf

        state = np.array(board.get_state())
        state = np.transpose(state, (1, 2, 0))          # (10,5,5) → (5,5,10)
        policy_logits, value = self.nn.predict(state, training=False)
        policy_logits = np.array(policy_logits).flatten()  # (1300,)
        value = float(value.numpy()[0][0])

        available_moves = board.get_available_moves()
        if not available_moves:
            return {}, value

        # Masquer les actions invalides + softmax
        masked = np.full(1300, -1e9)
        for action in available_moves:
            col, row = action.from_pos
            flat_idx = col * 5 * 52 + row * 52 + action.move_idx
            masked[flat_idx] = policy_logits[flat_idx]
        exp_x = np.exp(masked - masked.max())
        probs = exp_x / exp_x.sum()

        return {action: probs[action.from_pos[0] * 5 * 52
                               + action.from_pos[1] * 52
                               + action.move_idx]
                for action in available_moves}, value

    # ── Expansion ─────────────────────────────────────────────────────────────

    def _expand(self, node: MCTSNode, action_probs: dict,
                add_dirichlet: bool = False):
        """Crée les noeuds enfants avec les probabilités a priori du réseau."""
        if add_dirichlet and self.dir_alpha > 0 and action_probs:
            noise = np.random.dirichlet([self.dir_alpha] * len(action_probs))
            for i, (action, prob) in enumerate(action_probs.items()):
                p = (1 - self.dir_eps) * prob + self.dir_eps * noise[i]
                node.children[action] = MCTSNode(prior=p)
        else:
            for action, prob in action_probs.items():
                node.children[action] = MCTSNode(prior=prob)

    # ── Simulation ────────────────────────────────────────────────────────────

    def _simulate(self, root: MCTSNode, board: Board):
        """
        Une simulation MCTS complète :
        Sélection → Expansion → Évaluation → Rétropropagation
        Le Board est modifié puis restauré à l'identique.
        """
        node = root
        path = []          # (parent_node, action, child_node, last_move, is_default)

        # ── Sélection ────────────────────────────────────────────────────────
        while node.is_expanded():
            parent_N = node.N
            # Choisir l'enfant avec le meilleur score PUCT
            best_action = max(node.children,
                              key=lambda a: node.children[a].puct_score(parent_N, self.c_puct))
            child = node.children[best_action]

            # Jouer le coup sur le plateau (modifie le Board en place)
            last_move = board.play_move(action=best_action)
            path.append((node, best_action, child, last_move))
            node = child

            # Vérifier fin de partie après chaque coup
            ended, _ = board.game_has_ended()
            if ended:
                break

        # ── Évaluation ───────────────────────────────────────────────────────
        ended, winner = board.game_has_ended()

        if ended:
            # État terminal : valeur exacte
            # Après play_move, current_player est celui qui n'a PAS joué en dernier.
            # Si winner == current_player → rare, sinon → il a perdu
            value = 1.0 if winner == board.current_player else -1.0
        else:
            # Feuille non terminale : évaluer + expanser
            action_probs, value = self._nn_evaluate(board)
            if action_probs:
                self._expand(node, action_probs)
            # Si pas de coup dispo : le noeud reste non expansé (valeur réseau seulement)

        # ── Rétropropagation ─────────────────────────────────────────────────
        # Convention : 'value' est du point de vue du joueur courant AU NIVEAU DE LA FEUILLE.
        # En remontant, on alterne le signe à chaque niveau (jeu à somme nulle).
        for parent, action, child, last_move in reversed(path):
            value = -value                 # changer de perspective en montant
            child.W += value              # stocker du point de vue du parent
            child.N += 1
            board.cancel_last_move(last_move)

        root.N += 1

    # ── Interface principale ──────────────────────────────────────────────────

    def search(self, board: Board) -> dict:
        """
        Lance n_simulations depuis l'état courant du Board.
        Retourne {action: visit_count} pour toutes les actions à la racine.
        Le Board est rendu dans son état d'origine après l'appel.
        """
        root = MCTSNode(prior=1.0)
        root.N = 1  # éviter sqrt(0) lors de la première sélection

        # Expansion de la racine avec bruit Dirichlet (exploration à la racine)
        action_probs, value = self._nn_evaluate(board)
        if not action_probs:
            return {}
        self._expand(root, action_probs, add_dirichlet=True)

        for _ in range(self.n_simulations):
            self._simulate(root, board)

        return {action: child.N for action, child in root.children.items()}

In [ ]:
class MCTSPlayer(Player):
    """
    Joueur utilisant MCTS + réseau de neurones (Kamae3).
    Compatible avec Game et GameSession.
    """

    def __init__(self, nn_player: CNNPlayer_v6,
                 n_simulations: int = 100, c_puct: float = 1.5,
                 dirichlet_alpha: float = 0.0, dirichlet_eps: float = 0.0):
        super().__init__()
        self.name = f'MCTSPlayer(n={n_simulations})'
        self.mcts = MCTS(nn_player, n_simulations=n_simulations,
                         c_puct=c_puct,
                         dirichlet_alpha=dirichlet_alpha,
                         dirichlet_eps=dirichlet_eps)

    def play(self, board: Board):
        visit_counts = self.mcts.search(board)
        if not visit_counts:
            return None
        # Choisir l'action la plus visitée (politique greedy sur les visites)
        return max(visit_counts, key=visit_counts.get)

## Benchmark de vitesse

Mesure le temps moyen par coup en fonction du nombre de simulations.

In [ ]:
import matplotlib.pyplot as plt

# Charger le modèle une fois
nn = CNNPlayer_v6()
nn.load_weights(WEIGHTS_PATH)

# Créer un adversaire pour générer des états
opp = LookAheadHeuristicPlayer(heuristic_function='heuristic_defensive', max_depth=1)

# Collecter des états de jeu réels via une partie
print("Collecte d'états de jeu pour le benchmark...")
bench_states = []
game = Game(player_one=nn, player_two=opp, verbose=False)
board = game.board

# Rejouer quelques coups pour avoir des états variés
mcts_bench = MCTS(nn, n_simulations=1)  # juste pour accéder à _nn_evaluate

# Benchmark sur différentes valeurs de n_simulations
sim_values = [10, 25, 50, 100, 200, 400]
times = []

# Recréer un plateau frais pour le benchmark
bench_game = Game(player_one=nn, player_two=opp, verbose=False)
bench_board = bench_game.board

for n_sim in sim_values:
    mcts = MCTS(nn, n_simulations=n_sim, c_puct=C_PUCT)
    t0 = time.time()
    for _ in range(BENCH_N_MOVES):
        mcts.search(bench_board)
        # Jouer un coup pour changer l'état (avec l'opponent)
        moves = bench_board.get_available_moves()
        if not moves:
            bench_board.play_default_move()
        else:
            bench_board.play_move(moves[0])
        ended, _ = bench_board.game_has_ended()
        if ended:
            break
    elapsed = (time.time() - t0) / BENCH_N_MOVES * 1000
    times.append(elapsed)
    print(f'  n_sim={n_sim:4d} → {elapsed:.1f} ms/coup')

# Graphique
plt.figure(figsize=(7, 4))
plt.plot(sim_values, times, 'o-', color='steelblue', linewidth=2)
plt.axhline(1000, color='red', linestyle='--', alpha=0.5, label='1 sec/coup')
plt.axhline(500,  color='orange', linestyle='--', alpha=0.5, label='0.5 sec/coup')
plt.xlabel('Nombre de simulations')
plt.ylabel('Temps par coup (ms)')
plt.title('Temps MCTS en fonction du nombre de simulations')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Évaluation : CNN seul vs MCTS

Compare la force du réseau **avec** et **sans** MCTS contre LookAhead1 et LookAhead2.

In [ ]:
# Charger le modèle (si pas déjà fait)
nn = CNNPlayer_v6()
nn.load_weights(WEIGHTS_PATH)

# Joueurs
cnn_player   = nn
mcts_player  = MCTSPlayer(nn, n_simulations=N_SIMULATIONS, c_puct=C_PUCT)

lookahead1   = LookAheadHeuristicPlayer(heuristic_function='heuristic_defensive', max_depth=1)
lookahead2   = LookAheadHeuristicPlayer(heuristic_function='heuristic_defensive', max_depth=2)

print(f'Réseau : {WEIGHTS_PATH}')
print(f'MCTS   : {N_SIMULATIONS} simulations, c_puct={C_PUCT}')
print(f'Eval   : {EVAL_N_GAMES} parties par matchup')

In [ ]:
def evaluate(player, opponent, n_games, label):
    """Joue n_games parties et retourne le ratio de victoires."""
    session = GameSession(player_one=player, player_two=opponent, number_of_games=n_games)
    session.start()
    stats = session.getStats()
    wins   = stats.get('winP1', 0)
    losses = stats.get('winP2', 0)
    draws  = stats.get('draw', 0)
    ratio  = wins / n_games
    print(f'{label:45s}  W={wins:3d}  L={losses:3d}  D={draws:2d}  → {ratio:.1%}')
    return ratio

print('─' * 70)
print(f'  Matchup                                               W    L    D    Ratio')
print('─' * 70)

r1 = evaluate(cnn_player,  lookahead1, EVAL_N_GAMES, 'CNN seul       vs LookAhead1')
r2 = evaluate(mcts_player, lookahead1, EVAL_N_GAMES, f'MCTS n={N_SIMULATIONS:<4d}  vs LookAhead1')
print()
r3 = evaluate(cnn_player,  lookahead2, EVAL_N_GAMES, 'CNN seul       vs LookAhead2')
r4 = evaluate(mcts_player, lookahead2, EVAL_N_GAMES, f'MCTS n={N_SIMULATIONS:<4d}  vs LookAhead2')
print('─' * 70)

In [ ]:
# Résumé graphique
fig, ax = plt.subplots(figsize=(7, 4))

x       = np.arange(2)
labels  = ['vs LookAhead1', 'vs LookAhead2']
cnn_r   = [r1, r3]
mcts_r  = [r2, r4]
width   = 0.35

ax.bar(x - width/2, cnn_r,  width, label='CNN seul',          color='steelblue',  alpha=0.8)
ax.bar(x + width/2, mcts_r, width, label=f'MCTS n={N_SIMULATIONS}', color='darkorange', alpha=0.8)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='50%')

for i, (c, m) in enumerate(zip(cnn_r, mcts_r)):
    ax.text(i - width/2, c + 0.01, f'{c:.0%}', ha='center', fontsize=9)
    ax.text(i + width/2, m + 0.01, f'{m:.0%}', ha='center', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Ratio victoires')
ax.set_title(f'Kamae3 — CNN seul vs MCTS (n={N_SIMULATIONS} simulations)')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Optionnel : impact de n_simulations sur la force de jeu

Mesure le ratio de victoires vs LookAhead2 en fonction du nombre de simulations.

In [ ]:
SWEEP_SIMS   = [10, 25, 50, 100, 200]  # réduire si trop lent
SWEEP_GAMES  = 50                       # moins de parties pour aller plus vite
sweep_ratios = []

print('Sweep n_simulations vs LookAhead2 :')
for n_sim in SWEEP_SIMS:
    player = MCTSPlayer(nn, n_simulations=n_sim, c_puct=C_PUCT)
    opp    = LookAheadHeuristicPlayer(heuristic_function='heuristic_defensive', max_depth=2)
    ratio  = evaluate(player, opp, SWEEP_GAMES, f'  MCTS n={n_sim:<4d} vs LookAhead2')
    sweep_ratios.append(ratio)

plt.figure(figsize=(7, 4))
plt.plot(SWEEP_SIMS, sweep_ratios, 'o-', color='darkorange', linewidth=2)
plt.axhline(r3, color='steelblue', linestyle='--', label=f'CNN seul ({r3:.0%})')
plt.axhline(0.5, color='gray', linestyle=':', alpha=0.5, label='50%')
plt.xlabel('Nombre de simulations')
plt.ylabel('Ratio victoires vs LookAhead2')
plt.title('Force de jeu MCTS vs nombre de simulations')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()